# Retail Sales Data Cleaning & Quality Assessment

This project focuses on preparing raw retail sales, product and inventory datasets for further business analysis. The notebook documents the full data cleaning process, including data quality checks, missing value review, duplicate detection, standardization of key fields and final validation.

The cleaned datasets created here will be used in a separate business analysis project focused on revenue growth, pricing, product performance and inventory optimization.

## Project Overview

The raw data used in this project contains three connected datasets:

- sales orders,
- product information,
- inventory records.

Before performing any business analysis, the datasets needed to be reviewed and cleaned. The main focus was to check whether key fields such as order dates, product identifiers, prices, quantities, discounts, product categories and stock values were complete, consistent and reliable enough for further analysis.

This notebook documents the full data cleaning process step by step. Each cleaning decision is based on the role of the column in the planned analysis and is followed by a validation check.

## Cleaning Workflow

The cleaning process follows a structured workflow:

1. Load the raw datasets.
2. Review dataset structure, column names and data types.
3. Identify missing values, duplicates and invalid records.
4. Standardize date columns and categorical fields.
5. Validate numerical columns used in revenue and inventory calculations.
6. Create cleaned versions of the datasets.
7. Run final quality checks.
8. Export cleaned files for further analysis.

The purpose of this notebook is not to draw business conclusions, but to prepare reliable data for the next stage of analysis.

In [155]:
from pathlib import Path

import numpy as np
import pandas as pd

In [156]:
from pathlib import Path

# Define project paths for Google Colab environment
RAW_DATA_PATH = Path("data/raw")
PROCESSED_DATA_PATH = Path("data/processed")

# Create project folders
RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

In [157]:
import shutil

raw_files = [
    "sales_orders.csv",
    "products.csv",
    "inventory.csv"
]

for file_name in raw_files:
    source_path = Path(file_name)
    target_path = RAW_DATA_PATH / file_name

    if source_path.exists() and not target_path.exists():
        shutil.move(source_path, target_path)

print("Raw files moved to data/raw folder.")

Raw files moved to data/raw folder.


In [158]:
# Load raw datasets
sales_orders_raw = pd.read_csv(RAW_DATA_PATH / "sales_orders.csv")
products_raw = pd.read_csv(RAW_DATA_PATH / "products.csv")
inventory_raw = pd.read_csv(RAW_DATA_PATH / "inventory.csv")

In [159]:
# Create working copies
sales_orders = sales_orders_raw.copy()
products = products_raw.copy()
inventory = inventory_raw.copy()

In [160]:
sales_orders_clean = sales_orders.copy()

sales_orders_completed = (
    sales_orders_clean[
        sales_orders_clean["status"] != "CANCELLED"
    ]
    .copy()
)

Cancelled orders were not treated as data errors. They were kept in the cleaned sales orders dataset, but a separate completed-orders dataset was created for future revenue analysis. This allows the analysis stage to exclude cancelled transactions without losing information about the original order status.

# Sales Orders Dataset — Data Quality Assessment & Cleaning

This section prepares the transaction-level sales data for further analysis. The dataset is reviewed for missing values, duplicate rows, date consistency, country naming issues, numerical validity and order status standardization.

The goal is to make sure that sales records are reliable before they are used for revenue, customer and product-level analysis.

## Dataset Overview

The first step is to review the structure of the raw sales orders dataset. This includes checking the number of records, available columns, data types and a small sample of rows.

This helps identify which columns require cleaning before the dataset can be used for analysis.

In [161]:
sales_orders.head()

,order_id,order_date,customer_id,country,product_id,quantity,unit_price,discount_pct,status
0,1,2023-04-25,54675,France,2196,6,149.95,30.27,Shipped
1,2,2022-06-20,10125,austria,268,6,144.87,0.0,COMPLETED
2,3,2020-12-11,33966,poland,964,5,191.27,NaN,DONE
3,4,2019/06/15,9665,italy,1637,2,174.30,26.53,COMPLETED
4,5,2022-08-19,21017,slovak,589,3,159.52,19.85,Done


In [162]:
sales_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260780 entries, 0 to 260779
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      260780 non-null  int64  
 1   order_date    260151 non-null  object 
 2   customer_id   260780 non-null  int64  
 3   country       260780 non-null  object 
 4   product_id    260780 non-null  int64  
 5   quantity      260780 non-null  int64  
 6   unit_price    260780 non-null  float64
 7   discount_pct  229458 non-null  object 
 8   status        260780 non-null  object 
dtypes: float64(1), int64(4), object(4)
memory usage: 17.9+ MB


In [163]:
sales_orders_initial_rows = len(sales_orders)
sales_orders_initial_columns = sales_orders.shape[1]

print(f"Initial number of rows: {sales_orders_initial_rows:,}")
print(f"Initial number of columns: {sales_orders_initial_columns}")

Initial number of rows: 260,780
Initial number of columns: 9


## Initial Data Quality Snapshot

Before cleaning individual columns, a general quality check was performed. This step summarizes the initial number of rows, duplicate records and missing values in the dataset.

In [164]:
initial_quality_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Duplicate rows",
        "Total missing values"
    ],
    "Value": [
        sales_orders.shape[0],
        sales_orders.shape[1],
        sales_orders.duplicated().sum(),
        sales_orders.isna().sum().sum()
    ]
})

initial_quality_summary

,Metric,Value
0,Rows,260780
1,Columns,9
2,Duplicate rows,780
3,Total missing values,31951


## Missing Values Review

Missing values were reviewed column by column to understand which fields were affected.

This check is important because missing values in key fields such as order date, product identifier, quantity or unit price could directly affect later calculations and time-based analysis.

In [165]:
missing_values = (
    sales_orders
    .isna()
    .sum()
    .reset_index()
    .rename(columns={
        "index": "column",
        0: "missing_count"
    })
)

missing_values["missing_pct"] = (
    missing_values["missing_count"] / len(sales_orders) * 100
).round(2)

missing_values = (
    missing_values
    .sort_values("missing_count", ascending=False)
)

missing_values

,column,missing_count,missing_pct
7,discount_pct,31322,12.01
1,order_date,629,0.24
0,order_id,0,0.00
3,country,0,0.00
2,customer_id,0,0.00
4,product_id,0,0.00
5,quantity,0,0.00
6,unit_price,0,0.00
8,status,0,0.00


Missing values were not removed automatically. Each affected column was reviewed based on its role in the planned analysis.

Missing order dates require further cleaning because records without valid dates cannot be used reliably in time-based analysis. Missing discount values are handled separately because they may represent unavailable discount information rather than invalid records.

## Duplicate Records Review

Duplicate rows were checked to make sure the same transaction was not counted more than once. Technical duplicates can overstate order volume, revenue and product-level performance, so they need to be removed before further analysis.

In [166]:
duplicate_count = sales_orders.duplicated().sum()
duplicate_pct = duplicate_count / len(sales_orders) * 100

print(f"Duplicate rows: {duplicate_count:,}")
print(f"Duplicate share: {duplicate_pct:.2f}%")

Duplicate rows: 780
Duplicate share: 0.30%


In [167]:
sales_orders[
    sales_orders.duplicated(keep=False)
].sort_values("order_id").head(20)

,order_id,order_date,customer_id,country,product_id,quantity,unit_price,discount_pct,status
180,181,2020/08/12,43948,Czech Republic,1920,2,137.18,20.78,shipped
260542,181,2020/08/12,43948,Czech Republic,1920,2,137.18,20.78,shipped
260116,339,2024-12-15,59576,ES,748,4,186.33,19.27,done
338,339,2024-12-15,59576,ES,748,4,186.33,19.27,done
260619,586,2015-07-14,30922,GER,371,3,103.94,10.49,CANCELLED
585,586,2015-07-14,30922,GER,371,3,103.94,10.49,CANCELLED
260674,990,13/02/2022,27215,NL,1583,3,92.73,14.89,Shipped
989,990,13/02/2022,27215,NL,1583,3,92.73,14.89,Shipped
996,997,2021.02.14,29078,GER,1303,6,214.74,16.31,shipped
260062,997,2021.02.14,29078,GER,1303,6,214.74,16.31,shipped


The duplicated rows contained identical values across all columns, which indicates technical duplicates rather than separate valid transactions. These records were removed to avoid double-counting.

In [168]:
rows_before = len(sales_orders)

sales_orders = (
    sales_orders
    .drop_duplicates()
    .copy()
)

rows_after = len(sales_orders)
removed_duplicates = rows_before - rows_after
removed_duplicates_pct = removed_duplicates / rows_before * 100

print(f"Rows before removing duplicates: {rows_before:,}")
print(f"Rows after removing duplicates: {rows_after:,}")
print(f"Duplicate rows removed: {removed_duplicates:,}")
print(f"Removed share: {removed_duplicates_pct:.2f}%")
print(f"Remaining duplicate rows: {sales_orders.duplicated().sum()}")

Rows before removing duplicates: 260,780
Rows after removing duplicates: 260,000
Duplicate rows removed: 780
Removed share: 0.30%
Remaining duplicate rows: 0


## Order Date Standardization

The **order_date column** was imported as text and contained inconsistent date formats. Since the analysis will use time-based metrics, this column needs to be converted into a proper datetime format.

Invalid or missing dates are reviewed separately because records without a reliable order date cannot be assigned to a specific month or year.

In [169]:
sales_orders["order_date"].sample(15, random_state=42)

,order_date
21138,2018-08-13
250486,2024.09.15
186188,2021/06/08
117856,2021/11/08
95555,2022/01/06
132042,2016-08-28
176217,2019-10-04
21772,2015/05/23
12091,2019-01-06
159981,2019/11/03


In [170]:
sales_orders["order_date"] = (
    sales_orders["order_date"]
    .astype(str)
    .str.strip()
    .str.replace("/", "-", regex=False)
    .str.replace(".", "-", regex=False)
)

sales_orders["order_date"] = pd.to_datetime(
    sales_orders["order_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)
date_quality_summary = pd.DataFrame({
    "Metric": [
        "Missing or invalid dates",
        "Missing or invalid dates (%)",
        "Minimum date",
        "Maximum date"
    ],
    "Value": [
        sales_orders["order_date"].isna().sum(),
        round(sales_orders["order_date"].isna().mean() * 100, 2),
        sales_orders["order_date"].min(),
        sales_orders["order_date"].max()
    ]
})

date_quality_summary

,Metric,Value
0,Missing or invalid dates,1892
1,Missing or invalid dates (%),0.73
2,Minimum date,2015-01-01 00:00:00
3,Maximum date,2024-12-31 00:00:00


Rows with missing or invalid order dates were removed because they could not be assigned to a reliable time period. Since the planned analysis includes monthly and yearly trends, keeping records without valid dates could lead to misleading time-based results.

In [171]:
rows_before = len(sales_orders)

sales_orders = (
    sales_orders
    .dropna(subset=["order_date"])
    .copy()
)

rows_after = len(sales_orders)
removed_date_rows = rows_before - rows_after
removed_date_pct = removed_date_rows / rows_before * 100

print(f"Rows before removing invalid dates: {rows_before:,}")
print(f"Rows after removing invalid dates: {rows_after:,}")
print(f"Rows removed: {removed_date_rows:,}")
print(f"Removed share: {removed_date_pct:.2f}%")
print(f"Remaining missing dates: {sales_orders['order_date'].isna().sum()}")
date_range_check = sales_orders[
    (sales_orders["order_date"] < "2015-01-01") |
    (sales_orders["order_date"] > "2024-12-31")
]

print(f"Records outside expected date range: {len(date_range_check):,}")

Rows before removing invalid dates: 260,000
Rows after removing invalid dates: 258,108
Rows removed: 1,892
Removed share: 0.73%
Remaining missing dates: 0
Records outside expected date range: 0


## Country Standardization

The **country** column contained different versions of the same country names, including abbreviations, local-language names and inconsistent capitalization.

Country names were standardized to one consistent format to support reliable grouping and market-level analysis in the next project.

In [172]:
sales_orders["country"].unique()

array(['France', 'austria', 'poland', 'italy', 'slovak', 'Austria',
       'france', 'ES', 'FR', 'GER', 'CZ', 'Holland', 'Italy', 'pl',
       'SLOVAKIA', 'IT', 'SE', 'Slovakia', 'Czechia', 'Spain', 'czechia',
       'Czech Republic', 'AT', 'Deutschland', 'DE', 'Polska', 'SK',
       'netherlands', 'Sweden', 'spain', 'Germany', 'NL', 'germany',
       'sweden', 'Netherlands', 'POL', 'Poland', 'Czech', 'PL'],
      dtype=object)

In [173]:
country_mapping = {
    "pl": "Poland",
    "pol": "Poland",
    "polska": "Poland",
    "poland": "Poland",

    "fr": "France",
    "france": "France",

    "it": "Italy",
    "italy": "Italy",

    "es": "Spain",
    "spain": "Spain",

    "ger": "Germany",
    "de": "Germany",
    "deutschland": "Germany",
    "germany": "Germany",

    "cz": "Czech Republic",
    "czech": "Czech Republic",
    "czechia": "Czech Republic",
    "czech republic": "Czech Republic",

    "slovak": "Slovakia",
    "sk": "Slovakia",
    "slovakia": "Slovakia",

    "holland": "Netherlands",
    "nl": "Netherlands",
    "netherlands": "Netherlands",

    "at": "Austria",
    "austria": "Austria",

    "se": "Sweden",
    "sweden": "Sweden"
}

sales_orders["country"] = (
    sales_orders["country"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(country_mapping)
)

In [174]:
sales_orders["country"].value_counts().sort_index()

,count
country,
Austria,26229
Czech Republic,25568
France,25658
Germany,25828
Italy,25645
Netherlands,25953
Poland,25859
Slovakia,25980
Spain,25688


In [175]:
unexpected_countries = sales_orders[
    ~sales_orders["country"].isin(country_mapping.values())
]["country"].unique()

print(f"Unexpected country values: {unexpected_countries}")

Unexpected country values: []


## Quantity Validation

The **quantity** column was checked to identify zero, negative or otherwise invalid values. Since quantity is used to calculate sales volume and revenue, invalid quantities would directly affect the analysis.

In [176]:
sales_orders["quantity"].describe()

,quantity
count,258108.000000
mean,6.824155
std,46.766782
min,0.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,608.000000


In [177]:
invalid_quantity_mask = sales_orders["quantity"] <= 0
invalid_quantity_count = invalid_quantity_mask.sum()
invalid_quantity_pct = invalid_quantity_count / len(sales_orders) * 100

print(f"Invalid quantity records: {invalid_quantity_count:,}")
print(f"Invalid quantity share: {invalid_quantity_pct:.2f}%")
sales_orders.loc[
    invalid_quantity_mask,
    ["order_id", "order_date", "product_id", "quantity", "unit_price", "status"]
].head(20)
rows_before = len(sales_orders)

sales_orders = (
    sales_orders
    .loc[~invalid_quantity_mask]
    .copy()
)

rows_after = len(sales_orders)
removed_quantity_rows = rows_before - rows_after

print(f"Rows before quantity cleaning: {rows_before:,}")
print(f"Rows after quantity cleaning: {rows_after:,}")
print(f"Rows removed: {removed_quantity_rows:,}")
print(f"Remaining invalid quantities: {(sales_orders['quantity'] <= 0).sum()}")

Invalid quantity records: 1,027
Invalid quantity share: 0.40%
Rows before quantity cleaning: 258,108
Rows after quantity cleaning: 257,081
Rows removed: 1,027
Remaining invalid quantities: 0


## Unit Price Validation

The **unit_price** column was reviewed to identify zero or negative values. In a retail sales dataset, a unit price equal to or below zero is not suitable for revenue or pricing analysis unless there is a clear business explanation.

Because no separate explanation is available in the dataset, these records are treated as data quality issues.

In [178]:
sales_orders["unit_price"].describe()

,unit_price
count,257081.000000
mean,183.185825
std,80.636410
min,0.000000
25%,121.500000
50%,171.850000
75%,233.430000
max,769.720000


In [179]:
invalid_price_mask = sales_orders["unit_price"] <= 0
invalid_price_count = invalid_price_mask.sum()
invalid_price_pct = invalid_price_count / len(sales_orders) * 100

print(f"Invalid unit price records: {invalid_price_count:,}")
print(f"Invalid unit price share: {invalid_price_pct:.2f}%")

Invalid unit price records: 1,595
Invalid unit price share: 0.62%


In [180]:
sales_orders.loc[
    invalid_price_mask,
    ["order_id", "order_date", "product_id", "quantity", "unit_price", "discount_pct", "status"]
].head(20)
rows_before = len(sales_orders)

sales_orders = (
    sales_orders
    .loc[~invalid_price_mask]
    .copy()
)

rows_after = len(sales_orders)
removed_price_rows = rows_before - rows_after
removed_price_pct = removed_price_rows / rows_before * 100

print(f"Rows before unit price cleaning: {rows_before:,}")
print(f"Rows after unit price cleaning: {rows_after:,}")
print(f"Rows removed: {removed_price_rows:,}")
print(f"Removed share: {removed_price_pct:.2f}%")
print(f"Remaining invalid unit prices: {(sales_orders['unit_price'] <= 0).sum()}")

Rows before unit price cleaning: 257,081
Rows after unit price cleaning: 255,486
Rows removed: 1,595
Removed share: 0.62%
Remaining invalid unit prices: 0


## Discount Standardization

The **discount_pct** column was imported as text even though it represents numerical percentage values. Some values also contained the % symbol.

The column was standardized by removing percentage symbols and converting the values to a numeric format. Missing values were preserved because the dataset does not provide enough information to confirm whether they mean no discount or unavailable discount information.

In [181]:
sales_orders["discount_pct"].dropna().sample(15, random_state=42)

,discount_pct
204703,4.43
165627,0.0
97019,22.4
212982,6.6
186627,0.0
48179,12.59
90013,14.41
103264,21.78
221098,6.19
211462,1.76


In [182]:
discount_missing_before = sales_orders["discount_pct"].isna().sum()

sales_orders["discount_pct"] = (
    sales_orders["discount_pct"]
    .astype(str)
    .str.strip()
    .str.replace("%", "", regex=False)
    .replace({
        "nan": np.nan,
        "None": np.nan,
        "": np.nan
    })
)

sales_orders["discount_pct"] = pd.to_numeric(
    sales_orders["discount_pct"],
    errors="coerce"
)

discount_missing_after = sales_orders["discount_pct"].isna().sum()
new_missing_after_conversion = discount_missing_after - discount_missing_before

print(f"Missing discount values before conversion: {discount_missing_before:,}")
print(f"Missing discount values after conversion: {discount_missing_after:,}")
print(f"New missing values created during conversion: {new_missing_after_conversion:,}")
print(f"Discount column type: {sales_orders['discount_pct'].dtype}")

Missing discount values before conversion: 30,688
Missing discount values after conversion: 30,688
New missing values created during conversion: 0
Discount column type: float64


In [183]:
sales_orders["discount_pct"].describe()

,discount_pct
count,224798.000000
mean,12.554362
std,8.996327
min,0.000000
25%,5.310000
50%,11.940000
75%,18.730000
max,60.580000


In [184]:
invalid_discount_mask = (
    sales_orders["discount_pct"].notna() &
    ~sales_orders["discount_pct"].between(0, 100)
)

invalid_discount_count = invalid_discount_mask.sum()
invalid_discount_pct = invalid_discount_count / len(sales_orders) * 100

print(f"Invalid discount records: {invalid_discount_count:,}")
print(f"Invalid discount share: {invalid_discount_pct:.2f}%")

Invalid discount records: 0
Invalid discount share: 0.00%


In [185]:
sales_orders.loc[
    invalid_discount_mask,
    ["order_id", "order_date", "product_id", "unit_price", "discount_pct", "status"]
].head(20)
rows_before = len(sales_orders)

sales_orders = (
    sales_orders
    .loc[~invalid_discount_mask]
    .copy()
)

rows_after = len(sales_orders)
removed_discount_rows = rows_before - rows_after

print(f"Rows before discount validation: {rows_before:,}")
print(f"Rows after discount validation: {rows_after:,}")
print(f"Rows removed: {removed_discount_rows:,}")
print(
    "Remaining invalid discount values:",
    (
        sales_orders["discount_pct"].notna() &
        ~sales_orders["discount_pct"].between(0, 100)
    ).sum()
)

Rows before discount validation: 255,486
Rows after discount validation: 255,486
Rows removed: 0
Remaining invalid discount values: 0


## Order Status Standardization

The *status* column contained different labels for the same order status. These values were standardized into three consistent categories: COMPLETED, SHIPPED,CANCELED.

Cancelled orders were not treated as data errors. They were kept in the cleaned sales orders dataset because they are valid historical records. A separate revenue-ready dataset is created later for analysis that should exclude cancelled transactions.

In [186]:
sales_orders["status"].value_counts()
status_mapping = {
    "done": "COMPLETED",
    "complete": "COMPLETED",
    "completed": "COMPLETED",

    "ship": "SHIPPED",
    "shipped": "SHIPPED",

    "cancelled": "CANCELLED",
    "canceled": "CANCELLED"
}

sales_orders["status"] = (
    sales_orders["status"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(status_mapping)
)
sales_orders["status"].value_counts()
expected_statuses = {"COMPLETED", "SHIPPED", "CANCELLED"}

unexpected_statuses = sales_orders[
    ~sales_orders["status"].isin(expected_statuses)
]["status"].unique()

print(f"Unexpected status values: {unexpected_statuses}")

Unexpected status values: []


## Clean and Revenue-Ready Sales Orders Datasets

Two versions of the sales orders dataset were created:

**sales_orders_clean** — cleaned transaction data with all valid order statuses retained,

**sales_orders_revenue** — cleaned transaction data excluding cancelled orders, intended for future revenue analysis.

This separation keeps the cleaning stage transparent and avoids treating cancelled orders as data errors.

In [187]:
sales_orders_clean = sales_orders.copy()

sales_orders_revenue = (
    sales_orders_clean[
        sales_orders_clean["status"] != "CANCELLED"
    ]
    .copy()
)

In [188]:
sales_orders_dataset_summary = pd.DataFrame({
    "Dataset": [
        "sales_orders_clean",
        "sales_orders_revenue"
    ],
    "Rows": [
        len(sales_orders_clean),
        len(sales_orders_revenue)
    ],
    "Cancelled Orders": [
        (sales_orders_clean["status"] == "CANCELLED").sum(),
        (sales_orders_revenue["status"] == "CANCELLED").sum()
    ],
    "Duplicate Rows": [
        sales_orders_clean.duplicated().sum(),
        sales_orders_revenue.duplicated().sum()
    ]
})

sales_orders_dataset_summary

,Dataset,Rows,Cancelled Orders,Duplicate Rows
0,sales_orders_clean,255486,15457,0
1,sales_orders_revenue,240029,0,0


## Final Sales Orders Validation

The final validation checks whether the cleaned sales orders dataset is ready for the next stage of analysis. The checks focus on duplicate records, missing dates, invalid numerical values and unexpected categorical values.

In [189]:
final_sales_orders_validation = pd.DataFrame({
    "Check": [
        "Rows in initial dataset",
        "Rows in cleaned dataset",
        "Columns in cleaned dataset",
        "Duplicate rows",
        "Missing order dates",
        "Invalid quantities",
        "Invalid unit prices",
        "Invalid discount values",
        "Unexpected status values",
        "Unexpected country values"
    ],
    "Result": [
        sales_orders_initial_rows,
        len(sales_orders_clean),
        sales_orders_clean.shape[1],
        sales_orders_clean.duplicated().sum(),
        sales_orders_clean["order_date"].isna().sum(),
        (sales_orders_clean["quantity"] <= 0).sum(),
        (sales_orders_clean["unit_price"] <= 0).sum(),
        (
            sales_orders_clean["discount_pct"].notna() &
            ~sales_orders_clean["discount_pct"].between(0, 100)
        ).sum(),
        (~sales_orders_clean["status"].isin(expected_statuses)).sum(),
        (~sales_orders_clean["country"].isin(country_mapping.values())).sum()
    ]
})

final_sales_orders_validation

,Check,Result
0,Rows in initial dataset,260780
1,Rows in cleaned dataset,255486
2,Columns in cleaned dataset,9
3,Duplicate rows,0
4,Missing order dates,0
5,Invalid quantities,0
6,Invalid unit prices,0
7,Invalid discount values,0
8,Unexpected status values,0
9,Unexpected country values,0


In [190]:
sales_orders_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 255486 entries, 0 to 259998
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      255486 non-null  int64         
 1   order_date    255486 non-null  datetime64[ns]
 2   customer_id   255486 non-null  int64         
 3   country       255486 non-null  object        
 4   product_id    255486 non-null  int64         
 5   quantity      255486 non-null  int64         
 6   unit_price    255486 non-null  float64       
 7   discount_pct  224798 non-null  float64       
 8   status        255486 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(4), object(2)
memory usage: 19.5+ MB


In [191]:
sales_orders_clean.head()

,order_id,order_date,customer_id,country,product_id,quantity,unit_price,discount_pct,status
0,1,2023-04-25,54675,France,2196,6,149.95,30.27,SHIPPED
1,2,2022-06-20,10125,Austria,268,6,144.87,0.00,COMPLETED
2,3,2020-12-11,33966,Poland,964,5,191.27,NaN,COMPLETED
3,4,2019-06-15,9665,Italy,1637,2,174.30,26.53,COMPLETED
4,5,2022-08-19,21017,Slovakia,589,3,159.52,19.85,COMPLETED


## Sales Orders Cleaning Summary

The sales orders dataset required several cleaning steps before it could be used for further analysis. Duplicate records were removed, order dates were standardized, invalid or missing dates were removed, country names were unified, zero or negative numerical values were validated, discount values were converted to a numeric format and order statuses were standardized.
Cancelled orders were retained in the cleaned dataset because they are valid business records. For future revenue analysis, a separate **sales_orders_revenue** dataset was created with cancelled orders excluded.

The final validation confirmed that the cleaned sales orders dataset no longer contains duplicate rows, invalid dates, invalid quantities, invalid unit prices or unexpected status values.

# Products Dataset — Data Quality Assessment & Cleaning

This section prepares the product reference table used to describe products sold in the sales orders dataset. The checks focus on product identifier uniqueness, category consistency, base price validity and launch date standardization.

The goal is to ensure that product information can be safely joined with sales and inventory data without creating duplicated or incomplete records.

## Dataset Overview

The products dataset contains product-level information, including product identifiers, product categories, subcategories, base prices and launch dates.

Before using this dataset as a reference table, its structure was reviewed to check whether each product appears only once and whether key product attributes are complete and consistent.

In [192]:
products.head()

,product_id,category,sub_category,base_price,launch_date
0,1,Women,Activewear,163.08,2015-06-02
1,2,Shoes,Sneakers,194.81,2018-07-29
2,3,Shoes,Sandals,262.29,14-11-2018
3,4,Kids,Tops,108.43,2017-10-06
4,5,Kids,Tops,77.96,31-05-2017


In [193]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    2500 non-null   int64  
 1   category      2500 non-null   object 
 2   sub_category  2500 non-null   object 
 3   base_price    2500 non-null   float64
 4   launch_date   2409 non-null   object 
dtypes: float64(1), int64(1), object(3)
memory usage: 97.8+ KB


In [194]:
products_initial_rows = len(products)
products_initial_columns = products.shape[1]

print(f"Initial number of rows: {products_initial_rows:,}")
print(f"Initial number of columns: {products_initial_columns}")

Initial number of rows: 2,500
Initial number of columns: 5


## Initial Data Quality Snapshot

A general quality check was performed before reviewing individual columns. This step summarizes the number of records, columns, duplicate rows and missing values in the products dataset.

In [195]:
products_initial_quality = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Duplicate rows",
        "Total missing values"
    ],
    "Value": [
        products.shape[0],
        products.shape[1],
        products.duplicated().sum(),
        products.isna().sum().sum()
    ]
})

products_initial_quality

,Metric,Value
0,Rows,2500
1,Columns,5
2,Duplicate rows,0
3,Total missing values,91


## Product ID Validation

The **product_id** column was reviewed because it is the key field used to connect the products dataset with sales orders and inventory records.

Each product should have one unique product identifier. Duplicate or missing product IDs could lead to incorrect joins and duplicated product-level information.

In [196]:
product_id_summary = pd.DataFrame({
    "Metric": [
        "Missing product IDs",
        "Unique product IDs",
        "Total rows",
        "Duplicated product IDs"
    ],
    "Value": [
        products["product_id"].isna().sum(),
        products["product_id"].nunique(),
        len(products),
        products["product_id"].duplicated().sum()
    ]
})

product_id_summary

,Metric,Value
0,Missing product IDs,0
1,Unique product IDs,2500
2,Total rows,2500
3,Duplicated product IDs,0


In [197]:
products[
    products["product_id"].duplicated(keep=False)
].sort_values("product_id").head(20)

,product_id,category,sub_category,base_price,launch_date


The **product_id** column was kept as the product-level key. Records with missing or duplicated product identifiers would need to be reviewed because they could affect dataset joins.

## Missing Values Review

Missing values were reviewed column by column to understand which product attributes were affected.

Missing values in **product_id, category, sub_category or base_price** would directly affect joins, product grouping or pricing analysis. Missing **launch_date** values are reviewed separately because a product can still be valid even if its launch date is unavailable.

In [198]:
products_missing_values = (
    products
    .isna()
    .sum()
    .reset_index()
    .rename(columns={
        "index": "column",
        0: "missing_count"
    })
)

products_missing_values["missing_pct"] = (
    products_missing_values["missing_count"] / len(products) * 100
).round(2)

products_missing_values = (
    products_missing_values
    .sort_values("missing_count", ascending=False)
)

products_missing_values

,column,missing_count,missing_pct
4,launch_date,91,3.64
0,product_id,0,0.00
1,category,0,0.00
2,sub_category,0,0.00
3,base_price,0,0.00


Missing launch dates were not removed automatically. They were treated as unavailable product metadata, not as invalid product records. This avoids removing otherwise valid products from the reference table.

## Duplicate Records Review

Duplicate rows were checked to confirm that the product reference table does not contain repeated records. Duplicated products could create duplicated rows after joining with sales or inventory data.

In [199]:
products_duplicate_count = products.duplicated().sum()
products_duplicate_pct = products_duplicate_count / len(products) * 100

print(f"Duplicate rows: {products_duplicate_count:,}")
print(f"Duplicate share: {products_duplicate_pct:.2f}%")

Duplicate rows: 0
Duplicate share: 0.00%


In [200]:
products[
    products.duplicated(keep=False)
].sort_values("product_id").head(20)

,product_id,category,sub_category,base_price,launch_date


In [201]:
rows_before = len(products)

products = (
    products
    .drop_duplicates()
    .copy()
)

rows_after = len(products)
removed_product_duplicates = rows_before - rows_after
removed_product_duplicates_pct = removed_product_duplicates / rows_before * 100

print(f"Rows before removing duplicates: {rows_before:,}")
print(f"Rows after removing duplicates: {rows_after:,}")
print(f"Duplicate rows removed: {removed_product_duplicates:,}")
print(f"Removed share: {removed_product_duplicates_pct:.2f}%")
print(f"Remaining duplicate rows: {products.duplicated().sum()}")

Rows before removing duplicates: 2,500
Rows after removing duplicates: 2,500
Duplicate rows removed: 0
Removed share: 0.00%
Remaining duplicate rows: 0


## Category and Subcategory Standardization

The **category** and **sub_category** columns were reviewed for inconsistent capitalization, extra spaces and repeated labels written in different formats.

These fields are important because they will be used for product-level grouping in the analysis stage.

In [202]:
products[["category", "sub_category"]].head()

,category,sub_category
0,Women,Activewear
1,Shoes,Sneakers
2,Shoes,Sandals
3,Kids,Tops
4,Kids,Tops


In [203]:
products["category"].value_counts().sort_index()

,count
category,
Accessories,493
Kids,520
Men,475
Shoes,495
Women,517


In [204]:
products["sub_category"].value_counts().sort_index()

,count
sub_category,
Activewear,159
Bags,92
Belts,100
Boots,99
Bottoms,90
Dresses,82
Hats,110
Heels,90
Jeans,183


In [205]:
products["category"] = (
    products["category"]
    .astype(str)
    .str.strip()
    .str.title()
)

products["sub_category"] = (
    products["sub_category"]
    .astype(str)
    .str.strip()
    .str.title()
)

In [206]:
category_summary = pd.DataFrame({
    "Column": [
        "category",
        "sub_category"
    ],
    "Unique values": [
        products["category"].nunique(),
        products["sub_category"].nunique()
    ],
    "Missing values": [
        products["category"].isna().sum(),
        products["sub_category"].isna().sum()
    ]
})

In [207]:
category_summary
print("Categories:")
print(sorted(products["category"].unique()))

print("\nSubcategories:")
print(sorted(products["sub_category"].unique()))

Categories:
['Accessories', 'Kids', 'Men', 'Shoes', 'Women']

Subcategories:
['Activewear', 'Bags', 'Belts', 'Boots', 'Bottoms', 'Dresses', 'Hats', 'Heels', 'Jeans', 'Jewelry', 'Lingerie', 'Loafers', 'Outerwear', 'Sandals', 'Scarves', 'Shirts', 'Shoes', 'Sleepwear', 'Sneakers', 'Suits', 'T-Shirts', 'Tops']


## Base Price Validation

The **base_price** column was checked to identify missing, zero or negative values. Since this field represents product-level pricing information, invalid prices could affect pricing and product portfolio analysis.

In [208]:
products["base_price"].describe()

,base_price
count,2500.000000
mean,153.909268
std,61.142518
min,31.230000
25%,104.450000
50%,146.530000
75%,195.252500
max,409.970000


In [209]:
invalid_base_price_mask = products["base_price"].isna() | (products["base_price"] <= 0)

invalid_base_price_count = invalid_base_price_mask.sum()
invalid_base_price_pct = invalid_base_price_count / len(products) * 100

print(f"Invalid base price records: {invalid_base_price_count:,}")
print(f"Invalid base price share: {invalid_base_price_pct:.2f}%")

Invalid base price records: 0
Invalid base price share: 0.00%


In [210]:
products.loc[
    invalid_base_price_mask,
    ["product_id", "category", "sub_category", "base_price", "launch_date"]
].head(20)

,product_id,category,sub_category,base_price,launch_date


Products with missing, zero or negative base prices would not be reliable for future pricing analysis. If such records exist, they should be removed or reviewed before the dataset is exported.

In [211]:
rows_before = len(products)

products = (
    products
    .loc[~invalid_base_price_mask]
    .copy()
)

rows_after = len(products)
removed_invalid_price_products = rows_before - rows_after

print(f"Rows before base price validation: {rows_before:,}")
print(f"Rows after base price validation: {rows_after:,}")
print(f"Rows removed: {removed_invalid_price_products:,}")
print(f"Remaining invalid base prices: {(products['base_price'].isna() | (products['base_price'] <= 0)).sum()}")

Rows before base price validation: 2,500
Rows after base price validation: 2,500
Rows removed: 0
Remaining invalid base prices: 0


## Launch Date Standardization

The **launch_date** column contained dates stored in different text formats. The values were standardized by aligning date separators and converting the column to datetime.

Missing launch dates were preserved as NaT because the absence of a launch date does not make the entire product record invalid.

In [212]:
products["launch_date"].sample(15, random_state=42)

,launch_date
1447,2023-03-15
1114,2024-04-07
1064,2020/01/07
2287,2015-04-29
1537,2015-12-19
668,2017-08-03
1583,18-07-2020
2404,08-09-2016
497,2018/12/17
2480,2019/06/22


In [213]:
launch_date_missing_before = products["launch_date"].isna().sum()
launch_date_non_missing_before = products["launch_date"].notna().sum()

products["launch_date"] = (
    products["launch_date"]
    .astype(str)
    .str.strip()
    .str.replace("/", "-", regex=False)
    .str.replace(".", "-", regex=False)
    .replace({
        "nan": np.nan,
        "None": np.nan,
        "": np.nan
    })
)

products["launch_date"] = pd.to_datetime(
    products["launch_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

launch_date_missing_after = products["launch_date"].isna().sum()
new_invalid_launch_dates = launch_date_missing_after - launch_date_missing_before

print(f"Missing launch dates before conversion: {launch_date_missing_before:,}")
print(f"Non-missing launch dates before conversion: {launch_date_non_missing_before:,}")
print(f"Missing or invalid launch dates after conversion: {launch_date_missing_after:,}")
print(f"New invalid dates created during conversion: {new_invalid_launch_dates:,}")
print(f"Launch date column type: {products['launch_date'].dtype}")

Missing launch dates before conversion: 91
Non-missing launch dates before conversion: 2,409
Missing or invalid launch dates after conversion: 101
New invalid dates created during conversion: 10
Launch date column type: datetime64[ns]


In [214]:
launch_date_summary = pd.DataFrame({
    "Metric": [
        "Missing or invalid launch dates",
        "Missing or invalid launch dates (%)",
        "Minimum launch date",
        "Maximum launch date"
    ],
    "Value": [
        products["launch_date"].isna().sum(),
        round(products["launch_date"].isna().mean() * 100, 2),
        products["launch_date"].min(),
        products["launch_date"].max()
    ]
})

launch_date_summary

,Metric,Value
0,Missing or invalid launch dates,101
1,Missing or invalid launch dates (%),4.04
2,Minimum launch date,2015-01-02 00:00:00
3,Maximum launch date,2024-12-30 00:00:00


After conversion, launch dates are stored as datetime values. Missing or unparseable dates were left as NaT instead of removing the products, because these records can still be used in category, pricing and inventory analysis.

For analyses that specifically depend on launch timing, records with missing launch dates should be handled separately.

## Launch Year Creation

A **launch_year** column was created from the standardized **launch_date** field. This column will make it easier to use product launch timing in the analysis stage.

Products without a valid launch date keep a missing launch year.

In [215]:
products["launch_year"] = products["launch_date"].dt.year.astype("Int64")

In [216]:
products[["product_id", "launch_date", "launch_year"]].head(10)

,product_id,launch_date,launch_year
0,1,2015-06-02,2015
1,2,2018-07-29,2018
2,3,2018-11-14,2018
3,4,2017-10-06,2017
4,5,2017-05-31,2017
5,6,2020-01-13,2020
6,7,2018-10-09,2018
7,8,2017-12-11,2017
8,9,2019-02-17,2019
9,10,2018-06-09,2018


In [217]:
launch_year_summary = (
    products["launch_year"]
    .value_counts(dropna=False)
    .sort_index()
    .reset_index()
)

launch_year_summary.columns = ["launch_year", "product_count"]

launch_year_summary

,launch_year,product_count
0,2015,248
1,2016,253
2,2017,227
3,2018,236
4,2019,227
5,2020,235
6,2021,260
7,2022,239
8,2023,240
9,2024,234


## Product Coverage in Sales Orders

The products dataset was checked against the cleaned sales orders dataset to confirm whether all products appearing in sales orders are available in the product reference table.

This validation is important because missing product references could cause missing category, subcategory or base price information after joining datasets.

In [218]:
sales_product_ids = set(sales_orders_clean["product_id"].dropna().unique())
product_reference_ids = set(products["product_id"].dropna().unique())

missing_product_references = sales_product_ids - product_reference_ids
unused_product_references = product_reference_ids - sales_product_ids

print(f"Product IDs in sales orders: {len(sales_product_ids):,}")
print(f"Product IDs in products dataset: {len(product_reference_ids):,}")
print(f"Sales product IDs missing from products dataset: {len(missing_product_references):,}")
print(f"Products not present in sales orders: {len(unused_product_references):,}")

Product IDs in sales orders: 2,500
Product IDs in products dataset: 2,500
Sales product IDs missing from products dataset: 0
Products not present in sales orders: 0


In [219]:
if missing_product_references:
    display(
        sales_orders_clean[
            sales_orders_clean["product_id"].isin(missing_product_references)
        ][["order_id", "product_id", "order_date", "quantity", "unit_price"]]
        .head(20)
    )
else:
    print("All product IDs from sales orders are available in the products dataset.")

All product IDs from sales orders are available in the products dataset.


## Create Clean Products Dataset

A cleaned version of the products dataset was created after standardizing categories, validating prices and converting launch dates to datetime format.

In [220]:
products_clean = products.copy()

In [221]:
products_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2500 entries, 0 to 2499
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   product_id    2500 non-null   int64         
 1   category      2500 non-null   object        
 2   sub_category  2500 non-null   object        
 3   base_price    2500 non-null   float64       
 4   launch_date   2399 non-null   datetime64[ns]
 5   launch_year   2399 non-null   Int64         
dtypes: Int64(1), datetime64[ns](1), float64(1), int64(1), object(2)
memory usage: 139.2+ KB


In [222]:
products_clean.head()

,product_id,category,sub_category,base_price,launch_date,launch_year
0,1,Women,Activewear,163.08,2015-06-02,2015
1,2,Shoes,Sneakers,194.81,2018-07-29,2018
2,3,Shoes,Sandals,262.29,2018-11-14,2018
3,4,Kids,Tops,108.43,2017-10-06,2017
4,5,Kids,Tops,77.96,2017-05-31,2017


## Final Products Validation

The final validation checks whether the cleaned products dataset is ready to be used as a product reference table in the analysis stage.

In [223]:
final_products_validation = pd.DataFrame({
    "Check": [
        "Rows in initial dataset",
        "Rows in cleaned dataset",
        "Columns in cleaned dataset",
        "Duplicate rows",
        "Missing product IDs",
        "Duplicated product IDs",
        "Missing categories",
        "Missing subcategories",
        "Invalid base prices",
        "Missing or invalid launch dates",
        "Sales product IDs missing from products table"
    ],
    "Result": [
        products_initial_rows,
        len(products_clean),
        products_clean.shape[1],
        products_clean.duplicated().sum(),
        products_clean["product_id"].isna().sum(),
        products_clean["product_id"].duplicated().sum(),
        products_clean["category"].isna().sum(),
        products_clean["sub_category"].isna().sum(),
        (products_clean["base_price"].isna() | (products_clean["base_price"] <= 0)).sum(),
        products_clean["launch_date"].isna().sum(),
        len(missing_product_references)
    ]
})

final_products_validation

,Check,Result
0,Rows in initial dataset,2500
1,Rows in cleaned dataset,2500
2,Columns in cleaned dataset,6
3,Duplicate rows,0
4,Missing product IDs,0
5,Duplicated product IDs,0
6,Missing categories,0
7,Missing subcategories,0
8,Invalid base prices,0
9,Missing or invalid launch dates,101


## Products Cleaning Summary

The products dataset required mainly structural validation and date standardization. Product identifiers were checked for missing and duplicated values, duplicate rows were reviewed, product categories and subcategories were standardized, and base prices were validated.

The **launch_date** column contained mixed date formats and was converted to datetime. Missing or unparseable launch dates were kept as missing values because they do not make the full product record invalid.

A cleaned **products_clean** dataset was created for use in the next stage of the project.

# Inventory Dataset — Data Quality Assessment & Cleaning

This section prepares the inventory dataset, which contains stock information by product and warehouse country. The checks focus on product identifiers, missing values, duplicate records, warehouse country standardization, stock quantity validation and stock update dates.

The goal is to make sure that inventory records can be reliably connected with product data and used in the later inventory analysis.

## Dataset Overview

The inventory dataset contains stock-level information for products stored across different warehouse countries.

Before using this dataset in the analysis stage, its structure was reviewed to check whether product identifiers are valid, warehouse locations are consistent and stock quantities are suitable for inventory analysis.

In [224]:
inventory.head()

,product_id,warehouse_country,stock_quantity,last_stock_update
0,1,Germany,430,11-10-2024
1,1,Czech,129,2024-09-17
2,2,Polska,0,2024-07-06
3,2,Czechia,143,2023-10-13
4,3,GER,46,2024-01-25


In [225]:
inventory.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3741 entries, 0 to 3740
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   product_id         3741 non-null   int64 
 1   warehouse_country  3741 non-null   object
 2   stock_quantity     3741 non-null   int64 
 3   last_stock_update  3735 non-null   object
dtypes: int64(2), object(2)
memory usage: 117.0+ KB


In [226]:
inventory_initial_rows = len(inventory)
inventory_initial_columns = inventory.shape[1]

print(f"Initial number of rows: {inventory_initial_rows:,}")
print(f"Initial number of columns: {inventory_initial_columns}")

Initial number of rows: 3,741
Initial number of columns: 4


## Initial Data Quality Snapshot

A general quality check was performed before reviewing individual columns. This step summarizes the number of records, columns, duplicate rows and missing values in the inventory dataset.

In [227]:
inventory_initial_quality = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Duplicate rows",
        "Total missing values"
    ],
    "Value": [
        inventory.shape[0],
        inventory.shape[1],
        inventory.duplicated().sum(),
        inventory.isna().sum().sum()
    ]
})

inventory_initial_quality

,Metric,Value
0,Rows,3741
1,Columns,4
2,Duplicate rows,0
3,Total missing values,6


## Product ID Validation

The **product_id** column was reviewed because it is the key field used to connect inventory records with the products dataset.

Missing or invalid product identifiers could make it impossible to assign stock quantities to the correct product.

In [228]:
inventory_product_id_summary = pd.DataFrame({
    "Metric": [
        "Missing product IDs",
        "Unique product IDs",
        "Total inventory rows"
    ],
    "Value": [
        inventory["product_id"].isna().sum(),
        inventory["product_id"].nunique(),
        len(inventory)
    ]
})

inventory_product_id_summary

,Metric,Value
0,Missing product IDs,0
1,Unique product IDs,2500
2,Total inventory rows,3741


In [229]:
inventory[
    inventory["product_id"].isna()
].head(20)

,product_id,warehouse_country,stock_quantity,last_stock_update


The inventory dataset can contain multiple rows for the same product because one product may be stored in more than one warehouse country. Therefore, duplicated **product_id** values alone are not treated as an error.

## Missing Values Review

Missing values were reviewed column by column to identify which inventory fields were affected.

Missing values in **product_id**, **warehouse_country** or **stock_quantity** would directly affect inventory analysis. Missing **last_stock_update** values are reviewed separately because they describe metadata about stock freshness rather than the stock quantity itself.

In [230]:
inventory_missing_values = (
    inventory
    .isna()
    .sum()
    .reset_index()
    .rename(columns={
        "index": "column",
        0: "missing_count"
    })
)

inventory_missing_values["missing_pct"] = (
    inventory_missing_values["missing_count"] / len(inventory) * 100
).round(2)

inventory_missing_values = (
    inventory_missing_values
    .sort_values("missing_count", ascending=False)
)

inventory_missing_values

,column,missing_count,missing_pct
3,last_stock_update,6,0.16
0,product_id,0,0.00
1,warehouse_country,0,0.00
2,stock_quantity,0,0.00


Missing values were not removed automatically. Each affected column was reviewed based on its importance for future inventory analysis.

## Duplicate Records Review

Duplicate rows were checked to ensure that identical inventory records were not repeated. Full-row duplicates could overstate stock quantities if the dataset were aggregated later.

In [231]:
inventory_duplicate_count = inventory.duplicated().sum()
inventory_duplicate_pct = inventory_duplicate_count / len(inventory) * 100

print(f"Duplicate rows: {inventory_duplicate_count:,}")
print(f"Duplicate share: {inventory_duplicate_pct:.2f}%")

Duplicate rows: 0
Duplicate share: 0.00%


In [232]:
inventory[
    inventory.duplicated(keep=False)
].sort_values(["product_id", "warehouse_country"]).head(20)

,product_id,warehouse_country,stock_quantity,last_stock_update


In [233]:
rows_before = len(inventory)

inventory = (
    inventory
    .drop_duplicates()
    .copy()
)

rows_after = len(inventory)
removed_inventory_duplicates = rows_before - rows_after
removed_inventory_duplicates_pct = removed_inventory_duplicates / rows_before * 100

print(f"Rows before removing duplicates: {rows_before:,}")
print(f"Rows after removing duplicates: {rows_after:,}")
print(f"Duplicate rows removed: {removed_inventory_duplicates:,}")
print(f"Removed share: {removed_inventory_duplicates_pct:.2f}%")
print(f"Remaining duplicate rows: {inventory.duplicated().sum()}")

Rows before removing duplicates: 3,741
Rows after removing duplicates: 3,741
Duplicate rows removed: 0
Removed share: 0.00%
Remaining duplicate rows: 0


## Product and Warehouse Combination Check

Because the same product can appear in more than one warehouse country, the dataset was also checked for duplicated combinations of **product_id** and **warehouse_country**.

Each product-country combination should appear once. If the same product appears multiple times in the same warehouse country, stock quantity aggregation could be affected.

In [234]:
duplicate_product_warehouse_count = (
    inventory
    .duplicated(subset=["product_id", "warehouse_country"])
    .sum()
)

duplicate_product_warehouse_pct = (
    duplicate_product_warehouse_count / len(inventory) * 100
)

print(f"Duplicated product-country combinations: {duplicate_product_warehouse_count:,}")
print(f"Duplicated product-country share: {duplicate_product_warehouse_pct:.2f}%")

Duplicated product-country combinations: 0
Duplicated product-country share: 0.00%


In [235]:
inventory[
    inventory.duplicated(
        subset=["product_id", "warehouse_country"],
        keep=False
    )
].sort_values(["product_id", "warehouse_country"]).head(20)

,product_id,warehouse_country,stock_quantity,last_stock_update


This check is used only as a validation step. Records should be removed or aggregated only if duplicated product-country combinations are confirmed to represent repeated inventory entries rather than separate valid stock records.

## Warehouse Country Standardization

The **warehouse_country** column contained different versions of the same country names, including abbreviations, local-language names and inconsistent capitalization.

These values were standardized to one consistent format to support reliable grouping by warehouse country.

In [236]:
inventory["warehouse_country"].value_counts().sort_index()

,count
warehouse_country,
CZ,252
Czech,250
Czech Republic,228
Czechia,278
DE,225
Deutschland,228
GER,242
Germany,241
PL,201


In [237]:
warehouse_country_mapping = {
    "ger": "Germany",
    "de": "Germany",
    "deutschland": "Germany",
    "germany": "Germany",

    "pl": "Poland",
    "pol": "Poland",
    "polska": "Poland",
    "poland": "Poland",

    "cz": "Czech Republic",
    "czech": "Czech Republic",
    "czechia": "Czech Republic",
    "czech republic": "Czech Republic"
}

inventory["warehouse_country"] = (
    inventory["warehouse_country"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(warehouse_country_mapping)
)

In [238]:
inventory["warehouse_country"].value_counts().sort_index()

,count
warehouse_country,
Czech Republic,1271
Germany,1200
Poland,1270


In [239]:
expected_warehouse_countries = set(warehouse_country_mapping.values())

unexpected_warehouse_countries = inventory[
    ~inventory["warehouse_country"].isin(expected_warehouse_countries)
]["warehouse_country"].unique()

print(f"Unexpected warehouse country values: {unexpected_warehouse_countries}")

Unexpected warehouse country values: []


## Stock Quantity Validation

The **stock_quantity** column was checked for missing, negative or otherwise invalid values.

A stock quantity equal to zero can be a valid business value because it may indicate that a product is currently out of stock. Negative stock quantities would require further review because they are not suitable for standard inventory analysis.

In [240]:
inventory["stock_quantity"].describe()

,stock_quantity
count,3741.000000
mean,181.902165
std,112.103843
min,0.000000
25%,99.000000
50%,179.000000
75%,258.000000
max,567.000000


In [241]:
invalid_stock_quantity_mask = (
    inventory["stock_quantity"].isna() |
    (inventory["stock_quantity"] < 0)
)

invalid_stock_quantity_count = invalid_stock_quantity_mask.sum()
invalid_stock_quantity_pct = invalid_stock_quantity_count / len(inventory) * 100

print(f"Invalid stock quantity records: {invalid_stock_quantity_count:,}")
print(f"Invalid stock quantity share: {invalid_stock_quantity_pct:.2f}%")

Invalid stock quantity records: 0
Invalid stock quantity share: 0.00%


In [242]:
inventory.loc[
    invalid_stock_quantity_mask,
    ["product_id", "warehouse_country", "stock_quantity", "last_stock_update"]
].head(20)

,product_id,warehouse_country,stock_quantity,last_stock_update


In [243]:
rows_before = len(inventory)

inventory = (
    inventory
    .loc[~invalid_stock_quantity_mask]
    .copy()
)

rows_after = len(inventory)
removed_invalid_stock_rows = rows_before - rows_after

print(f"Rows before stock quantity validation: {rows_before:,}")
print(f"Rows after stock quantity validation: {rows_after:,}")
print(f"Rows removed: {removed_invalid_stock_rows:,}")
print(f"Remaining invalid stock quantities: {(inventory['stock_quantity'].isna() | (inventory['stock_quantity'] < 0)).sum()}")

Rows before stock quantity validation: 3,741
Rows after stock quantity validation: 3,741
Rows removed: 0
Remaining invalid stock quantities: 0


## Zero Stock Review

Rows with zero stock were reviewed separately. A zero value is not treated as a data quality issue because it can represent a valid out-of-stock situation.

These records are kept in the cleaned dataset.

In [244]:
zero_stock_count = (inventory["stock_quantity"] == 0).sum()
zero_stock_pct = zero_stock_count / len(inventory) * 100

print(f"Zero stock records: {zero_stock_count:,}")
print(f"Zero stock share: {zero_stock_pct:.2f}%")

Zero stock records: 255
Zero stock share: 6.82%


In [245]:
inventory.loc[
    inventory["stock_quantity"] == 0,
    ["product_id", "warehouse_country", "stock_quantity", "last_stock_update"]
].head(20)

,product_id,warehouse_country,stock_quantity,last_stock_update
2,2,Poland,0,2024-07-06
14,11,Czech Republic,0,2024/02/17
18,15,Germany,0,2024/11/27
43,33,Czech Republic,0,2023/12/14
99,73,Czech Republic,0,2024-08-01
101,75,Germany,0,2023-11-22
105,78,Czech Republic,0,2023/06/27
106,79,Czech Republic,0,2024-06-08
126,93,Czech Republic,0,2024.11.05
129,95,Czech Republic,0,2024-07-20


## Last Stock Update Standardization

The **last_stock_update** column contained dates stored in different text formats. These values were standardized by aligning separators and converting the column to datetime.

Missing or unparseable update dates were kept as NaT because they do not invalidate the stock quantity itself. However, they are tracked because they may matter in future analyses focused on stock freshness.

In [246]:
inventory["last_stock_update"].sample(15, random_state=42)

,last_stock_update
3396,2024-12-14
1114,2024-04-15
351,2023-07-30
1983,2023-09-21
2321,2024.03.14
2148,2023.08.20
93,2024-05-07
358,2023.12.01
3675,2024-04-10
1233,05/12/2024


In [247]:
stock_update_missing_before = inventory["last_stock_update"].isna().sum()
stock_update_non_missing_before = inventory["last_stock_update"].notna().sum()

inventory["last_stock_update"] = (
    inventory["last_stock_update"]
    .astype(str)
    .str.strip()
    .str.replace("/", "-", regex=False)
    .str.replace(".", "-", regex=False)
    .replace({
        "nan": np.nan,
        "None": np.nan,
        "": np.nan
    })
)

inventory["last_stock_update"] = pd.to_datetime(
    inventory["last_stock_update"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

stock_update_missing_after = inventory["last_stock_update"].isna().sum()
new_invalid_stock_update_dates = (
    stock_update_missing_after - stock_update_missing_before
)

print(f"Missing stock update dates before conversion: {stock_update_missing_before:,}")
print(f"Non-missing stock update dates before conversion: {stock_update_non_missing_before:,}")
print(f"Missing or invalid stock update dates after conversion: {stock_update_missing_after:,}")
print(f"New invalid dates created during conversion: {new_invalid_stock_update_dates:,}")
print(f"Last stock update column type: {inventory['last_stock_update'].dtype}")

Missing stock update dates before conversion: 6
Non-missing stock update dates before conversion: 3,735
Missing or invalid stock update dates after conversion: 19
New invalid dates created during conversion: 13
Last stock update column type: datetime64[ns]


In [248]:
stock_update_summary = pd.DataFrame({
    "Metric": [
        "Missing or invalid stock update dates",
        "Missing or invalid stock update dates (%)",
        "Minimum stock update date",
        "Maximum stock update date"
    ],
    "Value": [
        inventory["last_stock_update"].isna().sum(),
        round(inventory["last_stock_update"].isna().mean() * 100, 2),
        inventory["last_stock_update"].min(),
        inventory["last_stock_update"].max()
    ]
})

stock_update_summary

,Metric,Value
0,Missing or invalid stock update dates,19
1,Missing or invalid stock update dates (%),0.51
2,Minimum stock update date,2023-05-12 00:00:00
3,Maximum stock update date,2024-12-31 00:00:00


The **last_stock_update** column was successfully converted to datetime. Missing or unparseable values were retained as missing values instead of removing the inventory records.

This keeps valid stock quantities in the dataset while clearly documenting incomplete update-date metadata.

## Inventory Product Coverage

The inventory dataset was checked against the cleaned products dataset to confirm whether all inventory product IDs are available in the product reference table.

This validation is important because missing product references could prevent inventory records from being connected with product categories, subcategories and base prices.

In [249]:
inventory_product_ids = set(inventory["product_id"].dropna().unique())
product_reference_ids = set(products_clean["product_id"].dropna().unique())

inventory_ids_missing_from_products = inventory_product_ids - product_reference_ids
products_without_inventory = product_reference_ids - inventory_product_ids

print(f"Product IDs in inventory dataset: {len(inventory_product_ids):,}")
print(f"Product IDs in products dataset: {len(product_reference_ids):,}")
print(f"Inventory product IDs missing from products dataset: {len(inventory_ids_missing_from_products):,}")
print(f"Products without inventory records: {len(products_without_inventory):,}")

Product IDs in inventory dataset: 2,500
Product IDs in products dataset: 2,500
Inventory product IDs missing from products dataset: 0
Products without inventory records: 0


In [250]:
if inventory_ids_missing_from_products:
    display(
        inventory[
            inventory["product_id"].isin(inventory_ids_missing_from_products)
        ][["product_id", "warehouse_country", "stock_quantity", "last_stock_update"]]
        .head(20)
    )
else:
    print("All product IDs from inventory are available in the products dataset.")

All product IDs from inventory are available in the products dataset.


## Create Clean Inventory Dataset

A cleaned version of the inventory dataset was created after standardizing warehouse country names, validating stock quantities and converting stock update dates to datetime format.

In [251]:
inventory_clean = inventory.copy()

In [252]:
inventory_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3741 entries, 0 to 3740
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   product_id         3741 non-null   int64         
 1   warehouse_country  3741 non-null   object        
 2   stock_quantity     3741 non-null   int64         
 3   last_stock_update  3722 non-null   datetime64[ns]
dtypes: datetime64[ns](1), int64(2), object(1)
memory usage: 146.1+ KB


In [253]:
inventory_clean.head()

,product_id,warehouse_country,stock_quantity,last_stock_update
0,1,Germany,430,2024-10-11
1,1,Czech Republic,129,2024-09-17
2,2,Poland,0,2024-07-06
3,2,Czech Republic,143,2023-10-13
4,3,Germany,46,2024-01-25


## Final Inventory Validation

The final validation checks whether the cleaned inventory dataset is ready to be used in the next stage of analysis.

In [254]:
final_inventory_validation = pd.DataFrame({
    "Check": [
        "Rows in initial dataset",
        "Rows in cleaned dataset",
        "Columns in cleaned dataset",
        "Duplicate rows",
        "Missing product IDs",
        "Invalid stock quantities",
        "Zero stock records",
        "Missing warehouse countries",
        "Unexpected warehouse country values",
        "Missing or invalid stock update dates",
        "Inventory product IDs missing from products table"
    ],
    "Result": [
        inventory_initial_rows,
        len(inventory_clean),
        inventory_clean.shape[1],
        inventory_clean.duplicated().sum(),
        inventory_clean["product_id"].isna().sum(),
        (
            inventory_clean["stock_quantity"].isna() |
            (inventory_clean["stock_quantity"] < 0)
        ).sum(),
        (inventory_clean["stock_quantity"] == 0).sum(),
        inventory_clean["warehouse_country"].isna().sum(),
        (
            ~inventory_clean["warehouse_country"]
            .isin(expected_warehouse_countries)
        ).sum(),
        inventory_clean["last_stock_update"].isna().sum(),
        len(inventory_ids_missing_from_products)
    ]
})

final_inventory_validation

,Check,Result
0,Rows in initial dataset,3741
1,Rows in cleaned dataset,3741
2,Columns in cleaned dataset,4
3,Duplicate rows,0
4,Missing product IDs,0
5,Invalid stock quantities,0
6,Zero stock records,255
7,Missing warehouse countries,0
8,Unexpected warehouse country values,0
9,Missing or invalid stock update dates,19


## Inventory Cleaning Summary

The inventory dataset required several validation and standardization steps before it could be used for further analysis. Duplicate rows were checked, warehouse country names were standardized, stock quantities were validated and stock update dates were converted to datetime format.

Zero stock values were retained because they can represent valid out-of-stock situations. Missing or unparseable **last_stock_update** values were also retained because they do not invalidate the stock quantity itself.

A cleaned **inventory_clean** dataset was created for use in the next stage of the project.